# NB01: Setup and environment check

Verifies the runtime for the IHGAMP pipeline. Checks PyTorch with CUDA,
OpenSlide, OpenCV, available disk space, and creates the workspace tree.
Optionally runs a brief GPU smoke test.

**Required environment variables**:
- `WORKSPACE`: project root for outputs (default: `./workspace`)
- `WSI_ROOT`: TCGA diagnostic WSIs (required for NB02/NB03)
- `DDR_DIR`: TCGA DDR resource directory (required for NB05)

**Manuscript reference** (Supplementary Table S6): single NVIDIA RTX 4090
(24 GB VRAM), CUDA 12.1, mixed precision (AMP) enabled, TF32 enabled.
Total processing time for 19,996 slides: 143.7 hours.

In [ ]:
import os
import sys
import json
import shutil
import platform
from pathlib import Path
from datetime import datetime

# resolve workspace from env, default to local ./workspace
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
WSI_ROOT = os.environ.get("WSI_ROOT", "")
WSI_ROOT = Path(WSI_ROOT).resolve() if WSI_ROOT else None
DDR_DIR = os.environ.get("DDR_DIR", "")
DDR_DIR = Path(DDR_DIR).resolve() if DDR_DIR else None

# subdirectories used by downstream notebooks
SUBDIRS = ["coords", "embeddings", "labels", "models", "results", "runs", "figures"]

print(f"WORKSPACE : {WORKSPACE}")
print(f"WSI_ROOT  : {WSI_ROOT if WSI_ROOT else '(unset; required for NB02/NB03)'}")
print(f"DDR_DIR   : {DDR_DIR if DDR_DIR else '(unset; required for NB05)'}")


In [ ]:
# system info
def get_system_info():
    info = {
        "platform": platform.system(),
        "release": platform.release(),
        "python": sys.version.split()[0],
    }
    try:
        info["is_wsl"] = "microsoft" in Path("/proc/version").read_text().lower()
    except Exception:
        info["is_wsl"] = False
    return info

os_info = get_system_info()
print(json.dumps(os_info, indent=2))


In [ ]:
# torch + cuda
def check_torch():
    info = {"installed": False, "cuda_available": False}
    try:
        import torch
        info["installed"] = True
        info["version"] = torch.__version__
        info["cuda_available"] = torch.cuda.is_available()
        if info["cuda_available"]:
            d = torch.cuda.current_device()
            info["device"] = torch.cuda.get_device_name(d)
            info["vram_gb"] = round(torch.cuda.get_device_properties(d).total_memory / 1024**3, 2)
            info["cuda_version"] = torch.version.cuda
    except Exception as e:
        info["error"] = str(e)
    return info

torch_info = check_torch()
print(json.dumps(torch_info, indent=2))


In [ ]:
# openslide + opencv
def check_openslide():
    info = {"installed": False}
    try:
        import openslide
        info["installed"] = True
        info["version"] = getattr(openslide, "__version__", "unknown")
    except Exception as e:
        info["error"] = str(e)
    return info

def check_cv2():
    info = {"installed": False}
    try:
        import cv2
        info["installed"] = True
        info["version"] = cv2.__version__
    except Exception as e:
        info["error"] = str(e)
    return info

openslide_info = check_openslide()
cv2_info = check_cv2()
print("openslide:", openslide_info)
print("opencv   :", cv2_info)


In [ ]:
# disk-space estimate based on manuscript-reported pipeline
# 19,996 slides * 2,000 tiles * 512 dim * 2 bytes (fp16) ~= 41 GB embeddings
# coordinate CSVs (no pixels saved) are negligible
EXPECTED_SLIDES = 19996
TILES_PER_SLIDE = 2000
EMBED_DIM = 512
EMBED_DTYPE_BYTES = 2

est_emb_gb = EXPECTED_SLIDES * TILES_PER_SLIDE * EMBED_DIM * EMBED_DTYPE_BYTES / 1024**3
probe = WORKSPACE if WORKSPACE.exists() else WORKSPACE.parent
free_workspace_gb = shutil.disk_usage(probe).free / 1024**3

print(f"estimated tile embeddings : {est_emb_gb:7.1f} GB (fp16, 512-d, 19,996 slides)")
print(f"free at WORKSPACE         : {free_workspace_gb:7.1f} GB")


In [ ]:
# optional gpu smoke test (~5 seconds of fp16 matmul)
RUN_GPU_SMOKETEST = bool(int(os.environ.get("RUN_GPU_SMOKETEST", "1")))

def gpu_smoketest(seconds=5):
    try:
        import torch
        import time
        if not torch.cuda.is_available():
            return {"ran": False, "reason": "cuda_not_available"}
        torch.backends.cuda.matmul.allow_tf32 = True
        dev = torch.device("cuda:0")
        n = 4096
        a = torch.randn(n, n, device=dev, dtype=torch.float16)
        b = torch.randn(n, n, device=dev, dtype=torch.float16)
        torch.cuda.synchronize()
        t0 = time.time()
        iters = 0
        while time.time() - t0 < seconds:
            _ = torch.matmul(a, b)
            iters += 1
        torch.cuda.synchronize()
        return {"ran": True, "iters": iters, "secs": seconds}
    except Exception as e:
        return {"ran": False, "reason": str(e)}

smoke = gpu_smoketest() if RUN_GPU_SMOKETEST else {"ran": False, "reason": "skipped"}
print(json.dumps(smoke, indent=2))


In [ ]:
# create workspace tree
WORKSPACE.mkdir(parents=True, exist_ok=True)
for sd in SUBDIRS:
    (WORKSPACE / sd).mkdir(parents=True, exist_ok=True)
print(f"workspace tree at {WORKSPACE}:")
for sd in SUBDIRS:
    print(f"  {sd}/")


In [ ]:
# preflight gate: pass/fail decision
gate = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "workspace": str(WORKSPACE),
    "wsi_root": str(WSI_ROOT) if WSI_ROOT else None,
    "ddr_dir": str(DDR_DIR) if DDR_DIR else None,
    "os": os_info,
    "torch": torch_info,
    "openslide": openslide_info,
    "opencv": cv2_info,
    "gpu_smoketest": smoke,
    "estimates_gb": {
        "embeddings": round(est_emb_gb, 1),
        "free_workspace": round(free_workspace_gb, 1),
    },
    "pass": True,
    "notes": [],
}

if not torch_info.get("installed"):
    gate["pass"] = False
    gate["notes"].append("PyTorch not installed.")
elif not torch_info.get("cuda_available"):
    gate["pass"] = False
    gate["notes"].append("CUDA unavailable; pipeline requires GPU.")

if not openslide_info.get("installed"):
    gate["pass"] = False
    gate["notes"].append("openslide-python missing; required for WSI access.")

if free_workspace_gb < est_emb_gb + 50:
    gate["pass"] = False
    gate["notes"].append(
        f"low disk space: need >= {est_emb_gb + 50:.0f} GB, have {free_workspace_gb:.0f} GB"
    )

(WORKSPACE / "results").mkdir(parents=True, exist_ok=True)
(WORKSPACE / "results" / "preflight_gate.json").write_text(json.dumps(gate, indent=2))

print()
print(f"gate verdict: {'PASS' if gate['pass'] else 'FAIL'}")
for n in gate["notes"]:
    print(f"  - {n}")
print(f"report: {WORKSPACE / 'results' / 'preflight_gate.json'}")
